# Data Preprocessing

This notebook isolates the preprocessing logic into separate, readable cells so the cleaning decisions are easy to justify during evaluation.


## Notebook Setup

The next cells import the required libraries, apply the shared plotting style, and load the processed train/validation/test splits used for preprocessing review.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns


In [ ]:
from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, f1_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance


In [ ]:
plt.rcParams.update({
    'figure.dpi': 140,
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'axes.grid': True,
    'grid.alpha': 0.6,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

PALETTE = {
    'primary': '#58a6ff',
    'secondary': '#bc8cff',
    'accent': '#ff7b72',
    'success': '#3fb950',
    'warning': '#d29922',
    'gradient': ['#58a6ff', '#bc8cff', '#ff7b72', '#3fb950', '#d29922', '#f778ba'],
    'sequential': sns.color_palette('mako', 10),
    'diverging': sns.color_palette('coolwarm', 10),
}

SEED = 42
np.random.seed(SEED)
print('Environment ready for notebook execution.')


### Code Block Explanation
This cell loads the processed splits that will be inspected for missingness, outliers, encoding, and scaling behavior.


In [ ]:
# ── Load pre-processed train / val / test splits ────────────────
PROC = '../data/processed/'

train = pd.read_csv(PROC + 'train.csv')
val   = pd.read_csv(PROC + 'val.csv')
test  = pd.read_csv(PROC + 'test.csv')

print(f'Train: {train.shape[0]:,} rows × {train.shape[1]} cols')
print(f'Val:   {val.shape[0]:,} rows × {val.shape[1]} cols')
print(f'Test:  {test.shape[0]:,} rows × {test.shape[1]} cols')
print(f'\nTarget column: target_repeat_within_180d')
print(f'Target distribution (train):')
print(train['target_repeat_within_180d'].value_counts(normalize=True).round(4))

---
# 4 · Data Preprocessing
> We systematically address **missing values**, **outliers**, **categorical encoding**, and **feature scaling** — with mathematical justification for each decision driven by our EDA findings.

---

## 4.1 · Missing Value Analysis

### Code Block Explanation
This code counts missing values per feature and, when needed, creates a horizontal bar chart showing how much data is absent. The plot makes it easy to see which columns still need attention.


In [ ]:
# ── Missing value analysis ────────────────────────────────────────
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({
    'Feature': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing_df)*0.4)))
    bars = ax.barh(missing_df['Feature'], missing_df['Missing %'],
                   color=PALETTE['accent'], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Value Percentages', pad=15)
    for bar, pct in zip(bars, missing_df['Missing %']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('../final_outputs/missing_values.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(missing_df.to_string(index=False))
else:
    print('✅  No missing values detected in the pre-processed training data.')
    print('    Missing values were already handled during the data preparation pipeline.')

### 📊 Interpretation — Missing Values
The pre-processed dataset has already been cleaned. For reference, the imputation strategy applied during data preparation was:
- **Numerical nulls**: Imputed with **median** (robust to outliers) rather than mean
- **Categorical nulls**: Imputed with **mode** or a dedicated "unknown" category
- **Review text**: Missing text replaced with `__NO_REVIEW_TEXT__` sentinel, with `text_present=0` flag

> **Theoretical justification**: Median imputation preserves the central tendency without being distorted by extreme values (which we identified in Section 3.2). For the high-skew monetary features, mean imputation would overestimate typical values.

## 4.2 · Outlier Detection & Treatment

### Code Block Explanation
This cell applies the IQR rule to each selected feature and reports the lower fence, upper fence, and outlier rate. It is a numeric summary of how extreme the long-tail behavior is.


In [ ]:
# ── Outlier detection using IQR method ────────────────────────────
outlier_cols = ['total_price', 'total_freight', 'payment_value_total',
                'delivery_days', 'approval_lag_hours', 'product_weight_g_mean',
                'package_volume_cm3_mean']
outlier_cols = [c for c in outlier_cols if c in train.columns]

outlier_stats = []
for col in outlier_cols:
    s = train[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((s < lower) | (s > upper)).sum()
    outlier_stats.append({
        'Feature': col,
        'Q1': f'{Q1:.2f}', 'Q3': f'{Q3:.2f}', 'IQR': f'{IQR:.2f}',
        'Lower Fence': f'{lower:.2f}', 'Upper Fence': f'{upper:.2f}',
        'Outliers': n_outliers, 'Outlier %': f'{n_outliers/len(s)*100:.1f}%'
    })

outlier_df = pd.DataFrame(outlier_stats)
print(outlier_df.to_string(index=False))

### Code Block Explanation
These box plots visualize the outlier structure feature by feature. Each box summarizes the middle 50% of values, while the individual dots show unusually large or small observations.


In [ ]:
# ── Box plots for outlier visualisation ───────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.ravel()

for i, col in enumerate(outlier_cols):
    ax = axes[i]
    data = train[col].dropna()
    bp = ax.boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=PALETTE['primary'], alpha=0.6),
                    medianprops=dict(color=PALETTE['warning'], linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor=PALETTE['accent'],
                                   markersize=2, alpha=0.3))
    ax.set_title(col, fontsize=10)
    ax.set_xticklabels([])

# Hide unused axes
for j in range(len(outlier_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Detection via IQR Box Plots', fontsize=16, y=1.02,
             fontweight='bold', color=PALETTE['primary'])
plt.tight_layout()
plt.savefig('../final_outputs/outlier_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()

### 📊 Interpretation — Outlier Analysis
- **Monetary features** (`total_price`, `payment_value_total`) have significant right-tail outliers (5–15% of data) — these are legitimate high-value orders, not errors.
- **`delivery_days`**: Outliers represent unusually long deliveries, often caused by remote locations or logistics failures — **domain-meaningful signals**, not noise.
- **`approval_lag_hours`**: Extreme values indicate orders that took days to approve — potentially fraudulent or problematic orders.

> **Treatment strategy**: Rather than removing outliers (which would discard valuable information), we:
> 1. Use the **`log1p_*` transformed versions** which compress the tail (already available)
> 2. Apply **winsorisation** (clipping to 1st/99th percentile) for remaining raw features  
> 3. Use **tree-based models** as our primary algorithms, which are inherently robust to outliers

## 4.3 · Categorical Encoding

### Code Block Explanation
This cell identifies the categorical columns that require encoding and prints their cardinality. It helps justify why some columns are better suited to one-hot encoding and others to target encoding.


In [ ]:
# ── Identify categorical columns ──────────────────────────────────
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
# Exclude non-feature columns
cat_cols = [c for c in cat_cols if c not in ['customer_unique_id', 'order_id',
                                              'score_time', 'review_text']]
print('Categorical features to encode:')
for col in cat_cols:
    nunique = train[col].nunique()
    top = train[col].value_counts().head(3)
    print(f'  {col}: {nunique} unique — top 3: {dict(top)}')

### Code Block Explanation
This cell performs the actual categorical encoding. It uses one-hot encoding for low-cardinality payment type and target encoding for the higher-cardinality location and product-category columns.


In [ ]:
# ── Encoding strategy ────────────────────────────────────────────
# payment_type_mode: Low cardinality (4-5) → One-Hot Encoding
# product_category_main: Medium cardinality (~70) → Target Encoding
# seller_state_mode, customer_state: Low cardinality (27) → Target Encoding

target_col = 'target_repeat_within_180d'
global_mean = train[target_col].mean()

# One-hot for low cardinality
if 'payment_type_mode' in train.columns:
    train_ohe = pd.get_dummies(train['payment_type_mode'], prefix='pay_type', dummy_na=False)
    print(f'One-Hot: payment_type_mode → {train_ohe.shape[1]} columns: {list(train_ohe.columns)}')

# Target encoding for high-cardinality (computed on train, applied to all)
for col in ['product_category_main', 'seller_state_mode', 'customer_state']:
    if col in train.columns:
        # Calculate target mean per category on TRAIN data only
        mapping = train.groupby(col)[target_col].mean()
        train[f'{col}_target_enc'] = train[col].map(mapping).fillna(global_mean)
        val[f'{col}_target_enc'] = val[col].map(mapping).fillna(global_mean)
        test[f'{col}_target_enc'] = test[col].map(mapping).fillna(global_mean)
        print(f'Target Encoded: {col} → mean range [{mapping.min():.4f}, {mapping.max():.4f}]')

print('\n✅  Categorical encoding complete')


### Why Target Encoding?
For **high-cardinality categoricals** like `product_category_main` (~70 categories):
- **One-Hot Encoding** would create 70 sparse columns, increasing dimensionality and model complexity
- **Label Encoding** imposes an artificial ordinal relationship
- **Target Encoding** maps each category to its **mean target rate**, creating a single informative numeric feature that directly captures the category's predictive power

> ⚠️ **Leakage mitigation**: Target encoding is computed **only on training data** and applied to val/test. This prevents information from the validation/test sets from leaking into the encoding.

## 4.4 · Feature Scaling

### Code Block Explanation
This cell defines the numeric modeling features and standardizes them using only the training split. The printed means and standard deviations confirm that the scaled frame behaves as expected.


In [ ]:
# ── Feature scaling ──────────────────────────────────────────────
# Define the feature set for modelling
num_features = [
    'review_score', 'text_present', 'text_char_len', 'text_word_count',
    'exclamation_count', 'question_count',
    'log1p_total_price', 'log1p_total_freight', 'log1p_payment_value_total',
    'payment_installments_max', 'payment_records', 'payment_type_nunique',
    'item_count', 'seller_count', 'product_count',
    'same_state_seller_customer',
    'log1p_approval_lag_hours', 'delivery_days',
    'delivery_delay_days_clipped', 'late_delivery_flag',
    'freight_ratio', 'payment_gap',
    'log1p_product_weight_g_mean', 'log1p_package_volume_cm3_mean',
    'product_photos_qty_mean', 'product_description_lenght_mean',
    'purchase_month', 'purchase_quarter', 'weekend_purchase_flag',
]
num_features = [c for c in num_features if c in train.columns]

# Target-encoded categoricals
te_features = [c for c in train.columns if c.endswith('_target_enc')]

# Fit scaler on training data only
scaler = StandardScaler()
X_train_num = train[num_features].fillna(0)
scaler.fit(X_train_num)

# Transform all splits
X_train_scaled = pd.DataFrame(scaler.transform(X_train_num),
                              columns=num_features, index=train.index)
X_val_scaled = pd.DataFrame(scaler.transform(val[num_features].fillna(0)),
                            columns=num_features, index=val.index)
X_test_scaled = pd.DataFrame(scaler.transform(test[num_features].fillna(0)),
                             columns=num_features, index=test.index)

print(f'Scaled feature matrix: {X_train_scaled.shape}')
print(f'Mean (should be ~0): {X_train_scaled.mean().mean():.6f}')
print(f'Std  (should be ~1): {X_train_scaled.std().mean():.6f}')

### Why StandardScaler?
- **Logistic Regression** uses gradient descent, which converges faster when features are on the same scale  
- **Tree-based models** (RF, GBR) are **scale-invariant** — they won't benefit from scaling, but it doesn't hurt them either  
- StandardScaler centers to μ=0, σ=1, which is preferred over MinMaxScaler because:
  1. It preserves information about outliers (MinMax compresses everything into [0,1])
  2. It's more robust when train and test distributions differ slightly

> **Important**: The scaler is fit **only on training data** and applied to val/test to prevent data leakage.

---